# CS229 Lecture 4: Perceptron & Generalized Linear Models
## Practice Notebook

This notebook covers:
1. **Perceptron Algorithm** — implementation and visualization
2. **Exponential Family** — deriving distributions
3. **Generalized Linear Models (GLMs)** — deriving and implementing Linear Regression, Logistic Regression, and Softmax Regression from the GLM framework

---

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_blobs, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('All imports successful!')

ModuleNotFoundError: No module named 'sklearn'

---
# Part 1: The Perceptron Algorithm

## 1.1 Theory

The perceptron is a binary classifier that uses a **threshold activation function**:

$$h_\theta(x) = g(\theta^T x)$$

where:

$$g(z) = \begin{cases} 1 & \text{if } z \geq 0 \\ 0 & \text{if } z < 0 \end{cases}$$

### Update Rule:
$$\theta_j := \theta_j + \alpha \left(y^{(i)} - h_\theta(x^{(i)})\right) x_j^{(i)}$$

- If prediction is **correct**: no update (the error term is 0)
- If prediction is **wrong**: weights are adjusted

### Key Property: 
The Perceptron Convergence Theorem guarantees convergence if data is **linearly separable**.

## 1.2 Implementation

In [ ]:
class Perceptron:
    """
    Perceptron binary classifier.
    
    Activation function: g(z) = 1 if z >= 0, else 0
    Update rule: theta := theta + alpha * (y - h(x)) * x
    """
    
    def __init__(self, learning_rate=0.1, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.theta = None
        self.history = []  # track misclassifications per epoch
    
    def _activation(self, z):
        """Threshold activation: g(z) = 1 if z >= 0, else 0"""
        return np.where(z >= 0, 1, 0)
    
    def _add_intercept(self, X):
        """Add bias/intercept term (column of ones)"""
        return np.hstack([np.ones((X.shape[0], 1)), X])
    
    def fit(self, X, y):
        """Train the perceptron using the perceptron learning rule."""
        X = self._add_intercept(X)
        n_samples, n_features = X.shape
        
        # Initialize weights to zeros
        self.theta = np.zeros(n_features)
        self.history = []
        
        for epoch in range(self.n_iterations):
            misclassified = 0
            
            for i in range(n_samples):
                # Forward pass
                z = np.dot(self.theta, X[i])
                y_pred = self._activation(z)
                
                # Update rule: theta += alpha * (y - y_pred) * x
                error = y[i] - y_pred
                self.theta += self.learning_rate * error * X[i]
                
                if error != 0:
                    misclassified += 1
            
            self.history.append(misclassified)
            
            # Early stopping if perfectly classified
            if misclassified == 0:
                print(f'Converged at epoch {epoch + 1}')
                break
        
        if misclassified > 0:
            print(f'Did not fully converge after {self.n_iterations} epochs')
        
        return self
    
    def predict(self, X):
        """Predict class labels."""
        X = self._add_intercept(X)
        z = np.dot(X, self.theta)
        return self._activation(z)
    
    def accuracy(self, X, y):
        """Compute classification accuracy."""
        predictions = self.predict(X)
        return np.mean(predictions == y)

print('Perceptron class defined!')

## 1.3 Generate Linearly Separable Data & Train

In [ ]:
# Generate linearly separable data
X_perc, y_perc = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=2.0, random_state=42
)

# Train perceptron
perc = Perceptron(learning_rate=0.1, n_iterations=100)
perc.fit(X_perc, y_perc)

print(f'Training accuracy: {perc.accuracy(X_perc, y_perc):.4f}')
print(f'Learned weights (bias, w1, w2): {perc.theta}')

## 1.4 Visualize Decision Boundary

In [ ]:
def plot_decision_boundary_perceptron(X, y, model, title='Perceptron Decision Boundary'):
    """Plot data points and the perceptron's linear decision boundary."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # --- Plot 1: Decision boundary ---
    ax = axes[0]
    
    # Create mesh grid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='red', label='Class 0', edgecolors='k', s=50)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='blue', label='Class 1', edgecolors='k', s=50)
    ax.set_xlabel('$x_1$', fontsize=12)
    ax.set_ylabel('$x_2$', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend()
    
    # --- Plot 2: Convergence ---
    ax2 = axes[1]
    ax2.plot(range(1, len(model.history) + 1), model.history, 'b-o', markersize=3)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Misclassified Samples', fontsize=12)
    ax2.set_title('Perceptron Convergence', fontsize=14)
    
    plt.tight_layout()
    plt.show()

plot_decision_boundary_perceptron(X_perc, y_perc, perc)

## 1.5 Perceptron Limitation: Non-Linearly Separable Data (XOR)

The perceptron **cannot** solve XOR — a famous limitation that led to the first "AI winter."

In [ ]:
# XOR dataset
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])  # XOR labels

perc_xor = Perceptron(learning_rate=0.1, n_iterations=1000)
perc_xor.fit(X_xor, y_xor)

print(f'\nXOR accuracy: {perc_xor.accuracy(X_xor, y_xor):.4f}')
print(f'Predictions: {perc_xor.predict(X_xor)}')
print(f'True labels:  {y_xor}')
print('\n→ The perceptron FAILS on XOR because it is not linearly separable!')

## 1.6 Exercise: Perceptron on AND / OR gates

**Try it yourself!** The AND and OR functions *are* linearly separable. Verify the perceptron can learn them.

In [ ]:
# Exercise: AND gate
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])  # AND labels

# TODO: Train a perceptron on the AND gate and print the accuracy
# perc_and = Perceptron(learning_rate=0.1, n_iterations=100)
# perc_and.fit(X_and, y_and)
# print(f'AND accuracy: {perc_and.accuracy(X_and, y_and)}')
# print(f'Predictions: {perc_and.predict(X_and)}')

# Exercise: OR gate
X_or = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_or = np.array([0, 1, 1, 1])  # OR labels

# TODO: Train a perceptron on the OR gate and print the accuracy
# perc_or = Perceptron(learning_rate=0.1, n_iterations=100)
# perc_or.fit(X_or, y_or)
# print(f'OR accuracy: {perc_or.accuracy(X_or, y_or)}')
# print(f'Predictions: {perc_or.predict(X_or)}')

print('Uncomment the code above and run!')

---
# Part 2: The Exponential Family

## 2.1 Theory

A distribution belongs to the **exponential family** if it can be written as:

$$p(y; \eta) = b(y) \cdot \exp\left(\eta^T T(y) - a(\eta)\right)$$

| Component | Name | Role |
|-----------|------|------|
| $\eta$ | Natural parameter | Controls the distribution |
| $T(y)$ | Sufficient statistic | Usually $T(y) = y$ |
| $a(\eta)$ | Log-partition function | Normalizer; ensures distribution sums/integrates to 1 |
| $b(y)$ | Base measure | Scaling factor independent of $\eta$ |

### Key property of $a(\eta)$:
- $a'(\eta) = E[T(y)]$ — first derivative gives the mean
- $a''(\eta) = \text{Var}[T(y)]$ — second derivative gives the variance

## 2.2 Bernoulli as Exponential Family

$$p(y; \phi) = \phi^y(1-\phi)^{1-y}$$

Taking $\log$:
$$\log p = y \log\phi + (1-y)\log(1-\phi) = y\log\frac{\phi}{1-\phi} + \log(1-\phi)$$

So:
$$p(y;\phi) = \exp\left(y \cdot \underbrace{\log\frac{\phi}{1-\phi}}_{\eta} - \underbrace{\left(-\log(1-\phi)\right)}_{a(\eta)}\right)$$

Therefore:
- $\eta = \log\frac{\phi}{1-\phi}$ → **logit function**
- $\phi = \frac{1}{1 + e^{-\eta}}$ → **sigmoid function** (inverse of logit)
- $T(y) = y$
- $a(\eta) = \log(1 + e^\eta)$
- $b(y) = 1$

In [ ]:
# Verify Bernoulli as exponential family

def bernoulli_standard(y, phi):
    """Standard Bernoulli PMF: p(y;phi) = phi^y * (1-phi)^(1-y)"""
    return (phi ** y) * ((1 - phi) ** (1 - y))

def bernoulli_exp_family(y, phi):
    """Bernoulli in exponential family form."""
    eta = np.log(phi / (1 - phi))          # natural parameter (logit)
    T_y = y                                 # sufficient statistic
    a_eta = np.log(1 + np.exp(eta))         # log-partition function
    b_y = 1                                 # base measure
    return b_y * np.exp(eta * T_y - a_eta)

# Test with various y and phi values
print('Verifying Bernoulli — Standard vs Exponential Family Form:')
print(f'{"y":>3} {"phi":>6} {"Standard":>12} {"Exp Family":>12} {"Match?":>8}')
print('-' * 45)
for y_val in [0, 1]:
    for phi_val in [0.2, 0.5, 0.7, 0.9]:
        std = bernoulli_standard(y_val, phi_val)
        exp_fam = bernoulli_exp_family(y_val, phi_val)
        match = np.isclose(std, exp_fam)
        print(f'{y_val:>3} {phi_val:>6.1f} {std:>12.6f} {exp_fam:>12.6f} {str(match):>8}')

## 2.3 Gaussian as Exponential Family

For $y \sim \mathcal{N}(\mu, \sigma^2)$ with $\sigma^2 = 1$ (fixed):

$$p(y; \mu) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{(y-\mu)^2}{2}\right) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{y^2}{2}\right) \cdot \exp\left(\mu y - \frac{\mu^2}{2}\right)$$

Therefore:
- $\eta = \mu$
- $T(y) = y$
- $a(\eta) = \eta^2 / 2 = \mu^2 / 2$
- $b(y) = \frac{1}{\sqrt{2\pi}} \exp(-y^2/2)$

In [ ]:
# Verify Gaussian (sigma=1) as exponential family

def gaussian_standard(y, mu, sigma=1):
    """Standard Gaussian PDF."""
    return (1 / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-0.5 * ((y - mu) / sigma) ** 2)

def gaussian_exp_family(y, mu):
    """Gaussian (sigma=1) in exponential family form."""
    eta = mu                                          # natural parameter
    T_y = y                                           # sufficient statistic
    a_eta = eta ** 2 / 2                              # log-partition function
    b_y = (1 / np.sqrt(2 * np.pi)) * np.exp(-y**2 / 2)  # base measure
    return b_y * np.exp(eta * T_y - a_eta)

# Test
print('Verifying Gaussian (σ=1) — Standard vs Exponential Family Form:')
print(f'{"y":>6} {"mu":>6} {"Standard":>12} {"Exp Family":>12} {"Match?":>8}')
print('-' * 48)
for y_val in [-1.0, 0.0, 0.5, 2.0]:
    for mu_val in [0.0, 1.0, 2.0]:
        std = gaussian_standard(y_val, mu_val)
        exp_fam = gaussian_exp_family(y_val, mu_val)
        match = np.isclose(std, exp_fam)
        print(f'{y_val:>6.1f} {mu_val:>6.1f} {std:>12.6f} {exp_fam:>12.6f} {str(match):>8}')

## 2.4 Poisson as Exponential Family

$$p(y; \lambda) = \frac{\lambda^y e^{-\lambda}}{y!} = \frac{1}{y!} \exp(y \log\lambda - \lambda)$$

Therefore:
- $\eta = \log \lambda$ → $\lambda = e^\eta$
- $T(y) = y$
- $a(\eta) = e^\eta = \lambda$
- $b(y) = 1/y!$

In [ ]:
from math import factorial

def poisson_standard(y, lam):
    """Standard Poisson PMF."""
    return (lam ** y) * np.exp(-lam) / factorial(y)

def poisson_exp_family(y, lam):
    """Poisson in exponential family form."""
    eta = np.log(lam)
    T_y = y
    a_eta = np.exp(eta)  # = lambda
    b_y = 1 / factorial(y)
    return b_y * np.exp(eta * T_y - a_eta)

print('Verifying Poisson — Standard vs Exponential Family Form:')
print(f'{"y":>3} {"λ":>6} {"Standard":>12} {"Exp Family":>12} {"Match?":>8}')
print('-' * 45)
for y_val in [0, 1, 2, 5]:
    for lam_val in [1.0, 3.0, 5.0]:
        std = poisson_standard(y_val, lam_val)
        exp_fam = poisson_exp_family(y_val, lam_val)
        match = np.isclose(std, exp_fam)
        print(f'{y_val:>3} {lam_val:>6.1f} {std:>12.6f} {exp_fam:>12.6f} {str(match):>8}')

## 2.5 Visualizing the Log-Partition Function Property

Recall: $a'(\eta) = E[T(y)]$ and $a''(\eta) = \text{Var}[T(y)]$

Let's verify this for the Bernoulli distribution where $a(\eta) = \log(1 + e^\eta)$.

In [ ]:
eta_vals = np.linspace(-5, 5, 200)

# For Bernoulli: a(eta) = log(1 + exp(eta))
a_eta = np.log(1 + np.exp(eta_vals))

# First derivative: a'(eta) = sigmoid(eta) = E[y] = phi
a_prime = 1 / (1 + np.exp(-eta_vals))  # sigmoid

# Second derivative: a''(eta) = sigmoid(eta) * (1 - sigmoid(eta)) = Var[y] = phi(1-phi)
a_double_prime = a_prime * (1 - a_prime)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(eta_vals, a_eta, 'b-', linewidth=2)
axes[0].set_title('$a(\\eta) = \\log(1 + e^\\eta)$', fontsize=13)
axes[0].set_xlabel('$\\eta$')
axes[0].set_ylabel('$a(\\eta)$')

axes[1].plot(eta_vals, a_prime, 'r-', linewidth=2)
axes[1].set_title("$a'(\\eta) = \\sigma(\\eta) = E[y]$", fontsize=13)
axes[1].set_xlabel('$\\eta$')
axes[1].set_ylabel("$a'(\\eta)$")
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

axes[2].plot(eta_vals, a_double_prime, 'g-', linewidth=2)
axes[2].set_title("$a''(\\eta) = \\sigma(\\eta)(1-\\sigma(\\eta)) = Var[y]$", fontsize=13)
axes[2].set_xlabel('$\\eta$')
axes[2].set_ylabel("$a''(\\eta)$")

plt.suptitle('Bernoulli Log-Partition Function and Its Derivatives', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Key insight: The derivatives of the log-partition function give us')
print('the mean and variance of the distribution — this is true for ALL')
print('exponential family distributions!')

---
# Part 3: Generalized Linear Models (GLMs)

## 3.1 The GLM Framework

GLMs provide a **recipe** for constructing models. The three assumptions:

1. $y | x; \theta \sim \text{ExponentialFamily}(\eta)$
2. We want to predict $E[T(y) | x]$, i.e., $h_\theta(x) = E[T(y) | x]$
3. The natural parameter is linear in $x$: $\eta = \theta^T x$

### The GLM Recipe:
1. **Choose** an exponential family distribution for $y$
2. **Derive** the response (mean) function from the distribution
3. **Set** $\eta = \theta^T x$
4. The model and update rules follow automatically!

### The Universal Update Rule:
For ALL GLMs, the gradient ascent update has the same elegant form:
$$\theta_j := \theta_j + \alpha \left(y^{(i)} - h_\theta(x^{(i)})\right) x_j^{(i)}$$

## 3.2 GLM → Linear Regression (Gaussian)

**Choice:** $y | x; \theta \sim \mathcal{N}(\mu, 1)$

- Response function: $E[y|x] = \mu = \eta = \theta^T x$
- Therefore: $h_\theta(x) = \theta^T x$

This is just **ordinary least squares linear regression**!

In [ ]:
class GLM_LinearRegression:
    """
    Linear Regression derived from GLM with Gaussian distribution.
    
    Response function: h(x) = theta^T x  (identity)
    Loss: Mean Squared Error (from Gaussian MLE)
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.theta = None
        self.loss_history = []
    
    def _add_intercept(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])
    
    def _response_function(self, eta):
        """For Gaussian: response function is identity. E[y] = eta = theta^T x"""
        return eta
    
    def fit(self, X, y):
        """Train using batch gradient descent with the GLM update rule."""
        X = self._add_intercept(X)
        n_samples, n_features = X.shape
        self.theta = np.zeros(n_features)
        self.loss_history = []
        
        for iteration in range(self.n_iterations):
            # Forward pass
            eta = X @ self.theta
            h = self._response_function(eta)
            
            # GLM update: theta += alpha * X^T (y - h) / n
            error = y - h
            gradient = X.T @ error / n_samples
            self.theta += self.learning_rate * gradient
            
            # Track MSE loss
            mse = np.mean(error ** 2)
            self.loss_history.append(mse)
        
        return self
    
    def predict(self, X):
        X = self._add_intercept(X)
        eta = X @ self.theta
        return self._response_function(eta)

print('GLM Linear Regression class defined!')

In [ ]:
# Generate regression data
np.random.seed(42)
X_reg = 2 * np.random.randn(100, 1)
y_reg = 3 + 2.5 * X_reg.squeeze() + np.random.randn(100) * 0.8  # y = 3 + 2.5x + noise

# Train GLM Linear Regression
lr_model = GLM_LinearRegression(learning_rate=0.05, n_iterations=500)
lr_model.fit(X_reg, y_reg)

print(f'Learned parameters: bias = {lr_model.theta[0]:.4f}, slope = {lr_model.theta[1]:.4f}')
print(f'True parameters:    bias = 3.0000, slope = 2.5000')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regression fit
X_plot = np.linspace(X_reg.min() - 0.5, X_reg.max() + 0.5, 100).reshape(-1, 1)
y_plot = lr_model.predict(X_plot)

axes[0].scatter(X_reg, y_reg, c='steelblue', alpha=0.7, edgecolors='k', s=40, label='Data')
axes[0].plot(X_plot, y_plot, 'r-', linewidth=2, label=f'GLM fit: y = {lr_model.theta[0]:.2f} + {lr_model.theta[1]:.2f}x')
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('y', fontsize=12)
axes[0].set_title('GLM Linear Regression (Gaussian)', fontsize=14)
axes[0].legend(fontsize=11)

# Loss curve
axes[1].plot(lr_model.loss_history, 'b-', linewidth=1.5)
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('MSE Loss', fontsize=12)
axes[1].set_title('Training Loss', fontsize=14)

plt.tight_layout()
plt.show()

## 3.3 GLM → Logistic Regression (Bernoulli)

**Choice:** $y | x; \theta \sim \text{Bernoulli}(\phi)$

- Response function: $E[y|x] = \phi = \sigma(\eta) = \frac{1}{1 + e^{-\theta^T x}}$
- Therefore: $h_\theta(x) = \frac{1}{1 + e^{-\theta^T x}}$ — the **sigmoid** function!

Logistic regression **emerges naturally** from the GLM framework.

In [ ]:
class GLM_LogisticRegression:
    """
    Logistic Regression derived from GLM with Bernoulli distribution.
    
    Response function: h(x) = sigmoid(theta^T x)
    Loss: Binary Cross-Entropy (from Bernoulli MLE)
    """
    
    def __init__(self, learning_rate=0.1, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.theta = None
        self.loss_history = []
    
    def _add_intercept(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])
    
    def _response_function(self, eta):
        """For Bernoulli: response function is sigmoid. E[y] = 1/(1+exp(-eta))"""
        return 1 / (1 + np.exp(-np.clip(eta, -500, 500)))
    
    def _compute_loss(self, y, h):
        """Binary cross-entropy loss (negative log-likelihood of Bernoulli)."""
        epsilon = 1e-15
        h = np.clip(h, epsilon, 1 - epsilon)
        return -np.mean(y * np.log(h) + (1 - y) * np.log(1 - h))
    
    def fit(self, X, y):
        """Train using batch gradient ascent with the GLM update rule."""
        X = self._add_intercept(X)
        n_samples, n_features = X.shape
        self.theta = np.zeros(n_features)
        self.loss_history = []
        
        for iteration in range(self.n_iterations):
            # Forward pass
            eta = X @ self.theta
            h = self._response_function(eta)
            
            # GLM update: theta += alpha * X^T (y - h) / n
            # Note: SAME FORM as linear regression!
            error = y - h
            gradient = X.T @ error / n_samples
            self.theta += self.learning_rate * gradient
            
            # Track loss
            loss = self._compute_loss(y, h)
            self.loss_history.append(loss)
        
        return self
    
    def predict_proba(self, X):
        """Return predicted probabilities."""
        X = self._add_intercept(X)
        eta = X @ self.theta
        return self._response_function(eta)
    
    def predict(self, X, threshold=0.5):
        """Return predicted class labels."""
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)

print('GLM Logistic Regression class defined!')

In [ ]:
# Generate classification data
X_cls, y_cls = make_classification(
    n_samples=300, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.5, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X_cls, y_cls, test_size=0.3, random_state=42)

# Train GLM Logistic Regression
log_model = GLM_LogisticRegression(learning_rate=0.5, n_iterations=500)
log_model.fit(X_train, y_train)

print(f'Training accuracy: {log_model.accuracy(X_train, y_train):.4f}')
print(f'Test accuracy:     {log_model.accuracy(X_test, y_test):.4f}')
print(f'Learned theta: {log_model.theta}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision boundary
ax = axes[0]
x_min, x_max = X_cls[:, 0].min() - 1, X_cls[:, 0].max() + 1
y_min, y_max = X_cls[:, 1].min() - 1, X_cls[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = log_model.predict_proba(grid).reshape(xx.shape)

ax.contourf(xx, yy, Z, alpha=0.4, cmap='RdBu', levels=50)
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
ax.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1], c='red', label='Class 0', edgecolors='k', s=50)
ax.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1], c='blue', label='Class 1', edgecolors='k', s=50)
ax.set_xlabel('$x_1$', fontsize=12)
ax.set_ylabel('$x_2$', fontsize=12)
ax.set_title('GLM Logistic Regression (Bernoulli)', fontsize=14)
ax.legend()

# Loss curve
axes[1].plot(log_model.loss_history, 'b-', linewidth=1.5)
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[1].set_title('Training Loss', fontsize=14)

plt.tight_layout()
plt.show()

## 3.4 Comparing Update Rules: The Elegant Unification

Notice that **both** Linear Regression and Logistic Regression have the **same** update rule:

$$\theta := \theta + \alpha \cdot X^T(y - h_\theta(X)) / n$$

The only difference is the **response function** $h_\theta(x)$:

| Model | Distribution | Response Function $h_\theta(x)$ |
|-------|-------------|----------------------------------|
| Linear Regression | Gaussian | $\theta^T x$ (identity) |
| Logistic Regression | Bernoulli | $\sigma(\theta^T x)$ (sigmoid) |
| Poisson Regression | Poisson | $e^{\theta^T x}$ (exponential) |

This is the power of the GLM framework!

## 3.5 GLM → Poisson Regression

**Choice:** $y | x; \theta \sim \text{Poisson}(\lambda)$

- Response function: $E[y|x] = \lambda = e^{\eta} = e^{\theta^T x}$
- Useful for **count data** (e.g., number of website visits, number of events)

In [ ]:
class GLM_PoissonRegression:
    """
    Poisson Regression derived from GLM with Poisson distribution.
    
    Response function: h(x) = exp(theta^T x)
    Useful for count data.
    """
    
    def __init__(self, learning_rate=0.001, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.theta = None
        self.loss_history = []
    
    def _add_intercept(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])
    
    def _response_function(self, eta):
        """For Poisson: response function is exponential. E[y] = exp(eta)"""
        return np.exp(np.clip(eta, -20, 20))
    
    def fit(self, X, y):
        """Train using batch gradient descent with the GLM update rule."""
        X = self._add_intercept(X)
        n_samples, n_features = X.shape
        self.theta = np.zeros(n_features)
        self.loss_history = []
        
        for iteration in range(self.n_iterations):
            eta = X @ self.theta
            h = self._response_function(eta)
            
            # SAME update form: theta += alpha * X^T (y - h) / n
            error = y - h
            gradient = X.T @ error / n_samples
            self.theta += self.learning_rate * gradient
            
            # Poisson negative log-likelihood
            loss = np.mean(h - y * np.log(h + 1e-15))
            self.loss_history.append(loss)
        
        return self
    
    def predict(self, X):
        X = self._add_intercept(X)
        eta = X @ self.theta
        return self._response_function(eta)

# Generate count data
np.random.seed(42)
X_pois = np.random.uniform(0, 3, size=(200, 1))
true_lambda = np.exp(0.5 + 0.8 * X_pois.squeeze())  # true: intercept=0.5, slope=0.8
y_pois = np.random.poisson(lam=true_lambda)

# Train
pois_model = GLM_PoissonRegression(learning_rate=0.001, n_iterations=2000)
pois_model.fit(X_pois, y_pois)

print(f'Learned parameters: bias = {pois_model.theta[0]:.4f}, slope = {pois_model.theta[1]:.4f}')
print(f'True parameters:    bias = 0.5000, slope = 0.8000')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

X_plot_pois = np.linspace(0, 3, 100).reshape(-1, 1)
y_pred_pois = pois_model.predict(X_plot_pois)

axes[0].scatter(X_pois, y_pois, c='steelblue', alpha=0.5, edgecolors='k', s=30, label='Data (counts)')
axes[0].plot(X_plot_pois, y_pred_pois, 'r-', linewidth=2.5, label='GLM Poisson fit: $e^{\\theta^T x}$')
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('y (count)', fontsize=12)
axes[0].set_title('GLM Poisson Regression', fontsize=14)
axes[0].legend(fontsize=11)

axes[1].plot(pois_model.loss_history, 'b-', linewidth=1.5)
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Poisson NLL', fontsize=12)
axes[1].set_title('Training Loss', fontsize=14)

plt.tight_layout()
plt.show()

## 3.6 GLM → Softmax Regression (Multinomial)

**Choice:** $y | x; \theta \sim \text{Multinomial}$

For $k$ classes, the softmax function gives:

$$p(y = i | x; \theta) = \frac{e^{\theta_i^T x}}{\sum_{j=1}^{k} e^{\theta_j^T x}}$$

This generalizes logistic regression to **multi-class classification**.

### Note on over-parameterization:
We can set $\theta_k = \vec{0}$ (the last class as reference) since only differences between parameters matter.

In [ ]:
class GLM_SoftmaxRegression:
    """
    Softmax (Multinomial Logistic) Regression derived from GLM.
    
    Response function: softmax(theta^T x)
    Loss: Cross-Entropy (from Multinomial MLE)
    """
    
    def __init__(self, learning_rate=0.1, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.theta = None  # shape: (n_features, n_classes)
        self.loss_history = []
        self.n_classes = None
    
    def _add_intercept(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])
    
    def _softmax(self, Z):
        """Numerically stable softmax."""
        Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
        exp_Z = np.exp(Z_shifted)
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)
    
    def _one_hot_encode(self, y):
        """One-hot encode labels."""
        n_samples = len(y)
        one_hot = np.zeros((n_samples, self.n_classes))
        one_hot[np.arange(n_samples), y.astype(int)] = 1
        return one_hot
    
    def _compute_loss(self, Y_onehot, P):
        """Cross-entropy loss."""
        epsilon = 1e-15
        P = np.clip(P, epsilon, 1 - epsilon)
        return -np.mean(np.sum(Y_onehot * np.log(P), axis=1))
    
    def fit(self, X, y):
        """Train using batch gradient descent."""
        X = self._add_intercept(X)
        n_samples, n_features = X.shape
        self.n_classes = len(np.unique(y))
        
        # Initialize weight matrix: (n_features, n_classes)
        self.theta = np.zeros((n_features, self.n_classes))
        self.loss_history = []
        
        # One-hot encode labels
        Y_onehot = self._one_hot_encode(y)
        
        for iteration in range(self.n_iterations):
            # Forward pass: compute softmax probabilities
            Z = X @ self.theta  # (n_samples, n_classes)
            P = self._softmax(Z)
            
            # GLM update: theta += alpha * X^T (Y - P) / n
            # Same form! Error = (true - predicted)
            error = Y_onehot - P
            gradient = X.T @ error / n_samples
            self.theta += self.learning_rate * gradient
            
            loss = self._compute_loss(Y_onehot, P)
            self.loss_history.append(loss)
        
        return self
    
    def predict_proba(self, X):
        X = self._add_intercept(X)
        Z = X @ self.theta
        return self._softmax(Z)
    
    def predict(self, X):
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)
    
    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)

print('GLM Softmax Regression class defined!')

In [ ]:
# Generate multi-class data
X_multi, y_multi = make_blobs(
    n_samples=450, centers=3, n_features=2,
    cluster_std=1.2, random_state=42
)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.3, random_state=42
)

# Train Softmax Regression
softmax_model = GLM_SoftmaxRegression(learning_rate=0.5, n_iterations=500)
softmax_model.fit(X_train_m, y_train_m)

print(f'Training accuracy: {softmax_model.accuracy(X_train_m, y_train_m):.4f}')
print(f'Test accuracy:     {softmax_model.accuracy(X_test_m, y_test_m):.4f}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision regions
ax = axes[0]
x_min, x_max = X_multi[:, 0].min() - 1, X_multi[:, 0].max() + 1
y_min, y_max = X_multi[:, 1].min() - 1, X_multi[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = softmax_model.predict(grid).reshape(xx.shape)

ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
colors = ['red', 'blue', 'green']
for cls in range(3):
    mask = y_test_m == cls
    ax.scatter(X_test_m[mask, 0], X_test_m[mask, 1], c=colors[cls],
               label=f'Class {cls}', edgecolors='k', s=50)
ax.set_xlabel('$x_1$', fontsize=12)
ax.set_ylabel('$x_2$', fontsize=12)
ax.set_title('GLM Softmax Regression (Multinomial)', fontsize=14)
ax.legend()

# Loss curve
axes[1].plot(softmax_model.loss_history, 'b-', linewidth=1.5)
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[1].set_title('Training Loss', fontsize=14)

plt.tight_layout()
plt.show()

---
# Part 4: The Big Picture — GLM Unification

## 4.1 Comparing All Response Functions

In [ ]:
eta_range = np.linspace(-5, 5, 200)

# Response functions
identity = eta_range                          # Gaussian → Linear Regression
sigmoid = 1 / (1 + np.exp(-eta_range))        # Bernoulli → Logistic Regression
exponential = np.exp(eta_range)                # Poisson → Poisson Regression

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(eta_range, identity, 'b-', linewidth=2.5)
axes[0].set_title('Gaussian → Identity\n$h(x) = \\theta^T x$', fontsize=13)
axes[0].set_xlabel('$\\eta = \\theta^T x$', fontsize=12)
axes[0].set_ylabel('$E[y|x]$', fontsize=12)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)

axes[1].plot(eta_range, sigmoid, 'r-', linewidth=2.5)
axes[1].set_title('Bernoulli → Sigmoid\n$h(x) = \\frac{1}{1+e^{-\\theta^T x}}$', fontsize=13)
axes[1].set_xlabel('$\\eta = \\theta^T x$', fontsize=12)
axes[1].set_ylabel('$E[y|x]$', fontsize=12)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_ylim(-0.1, 1.1)

axes[2].plot(eta_range, exponential, 'g-', linewidth=2.5)
axes[2].set_title('Poisson → Exponential\n$h(x) = e^{\\theta^T x}$', fontsize=13)
axes[2].set_xlabel('$\\eta = \\theta^T x$', fontsize=12)
axes[2].set_ylabel('$E[y|x]$', fontsize=12)
axes[2].set_ylim(0, 30)

plt.suptitle('GLM Response Functions: Distribution → Model', fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

## 4.2 Summary Table

In [ ]:
summary_data = {
    'Problem Type': ['Regression', 'Binary Classification', 'Count Data', 'Multi-class Classification'],
    'Distribution': ['Gaussian', 'Bernoulli', 'Poisson', 'Multinomial'],
    'Response Function': ['Identity: η', 'Sigmoid: 1/(1+e^-η)', 'Exponential: e^η', 'Softmax: e^ηi / Σe^ηj'],
    'Model Name': ['Linear Regression', 'Logistic Regression', 'Poisson Regression', 'Softmax Regression'],
    'Natural Parameter η': ['μ', 'log(φ/(1-φ))', 'log(λ)', 'log(φi/φk)'],
    'Log-partition a(η)': ['η²/2', 'log(1+e^η)', 'e^η', 'log(Σe^ηi)']
}

# Print as formatted table
print('=' * 115)
print(f'{"GENERALIZED LINEAR MODELS — SUMMARY":^115}')
print('=' * 115)
headers = list(summary_data.keys())
widths = [25, 14, 22, 22, 18, 14]

header_line = ''
for h, w in zip(headers, widths):
    header_line += f'{h:<{w}}'
print(header_line)
print('-' * 115)

for i in range(4):
    row = ''
    for key, w in zip(headers, widths):
        row += f'{summary_data[key][i]:<{w}}'
    print(row)
print('=' * 115)
print()
print('KEY INSIGHT: All GLMs share the SAME update rule:')
print('  θ := θ + α · Xᵀ(y - h(x)) / n')
print('The ONLY difference is the response function h(x)!')

---
# Part 5: Exercises

## Exercise 1: Implement Perceptron with Different Learning Rates

Try different learning rates (0.01, 0.1, 1.0) and compare convergence speed.

In [ ]:
# TODO: Train perceptrons with different learning rates and plot convergence curves

# learning_rates = [0.01, 0.1, 1.0]
# for lr in learning_rates:
#     model = Perceptron(learning_rate=lr, n_iterations=100)
#     model.fit(X_perc, y_perc)
#     plt.plot(model.history, label=f'lr={lr}')
# plt.xlabel('Epoch')
# plt.ylabel('Misclassified')
# plt.legend()
# plt.title('Perceptron Convergence for Different Learning Rates')
# plt.show()

print('Uncomment and run the exercise above!')

## Exercise 2: Derive and Verify Another Exponential Family Distribution

The **Exponential distribution** with rate parameter $\lambda > 0$:

$$p(y; \lambda) = \lambda e^{-\lambda y}, \quad y \geq 0$$

**Task:** Write it in exponential family form and identify $\eta$, $T(y)$, $a(\eta)$, and $b(y)$.

*Hint:* $p(y;\lambda) = 1 \cdot \exp((-\lambda)y - (-\log\lambda))$

In [ ]:
# TODO: Implement and verify
# η = -λ  (so λ = -η, and η < 0)
# T(y) = y
# a(η) = -log(-η) = -log(λ)
# b(y) = 1 (for y >= 0)

# def exponential_standard(y, lam):
#     return lam * np.exp(-lam * y)

# def exponential_exp_family(y, lam):
#     eta = -lam
#     T_y = y
#     a_eta = -np.log(-eta)
#     b_y = 1
#     return b_y * np.exp(eta * T_y - a_eta)

# # Test
# for y_val in [0.5, 1.0, 2.0]:
#     for lam_val in [0.5, 1.0, 2.0]:
#         std = exponential_standard(y_val, lam_val)
#         exp_fam = exponential_exp_family(y_val, lam_val)
#         print(f'y={y_val}, λ={lam_val}: standard={std:.6f}, exp_family={exp_fam:.6f}, match={np.isclose(std, exp_fam)}')

print('Uncomment and run the exercise above!')

## Exercise 3: Logistic Regression vs Perceptron

Compare the decision boundaries of the Perceptron and GLM Logistic Regression on the same dataset.

In [ ]:
# TODO: Train both models on X_perc, y_perc and plot their decision boundaries side by side

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Perceptron
# perc_comp = Perceptron(learning_rate=0.1, n_iterations=100)
# perc_comp.fit(X_perc, y_perc)

# # Logistic Regression
# log_comp = GLM_LogisticRegression(learning_rate=0.5, n_iterations=500)
# log_comp.fit(X_perc, y_perc)

# # Plot both decision boundaries and compare
# # ... (use contourf and scatter as shown earlier)

print('Uncomment and implement the comparison!')
print('Question to think about: How do the decision boundaries differ?')
print('Which model gives probability estimates? Which only gives hard predictions?')

## Exercise 4: Softmax Regression on Iris Dataset

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

# Load Iris dataset
iris = load_iris()
X_iris = iris.data[:, :2]  # Use only first 2 features for visualization
y_iris = iris.target

# Standardize features
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# TODO: Train GLM_SoftmaxRegression on the Iris dataset
# TODO: Compute and print accuracy
# TODO: Visualize the decision boundaries

# iris_model = GLM_SoftmaxRegression(learning_rate=0.5, n_iterations=1000)
# iris_model.fit(X_iris_scaled, y_iris)
# print(f'Iris accuracy: {iris_model.accuracy(X_iris_scaled, y_iris):.4f}')

print('Uncomment and run!')
print(f'Classes: {iris.target_names}')
print(f'Features used: {iris.feature_names[:2]}')

---
# Part 6: Key Takeaways

## What we learned in this notebook:

### 1. Perceptron
- Simplest neural network: threshold activation + linear combination
- Converges only on **linearly separable** data
- Cannot solve XOR (historical limitation)
- Foundation for understanding more complex neural networks

### 2. Exponential Family
- A broad family that includes Gaussian, Bernoulli, Poisson, Multinomial, and many more
- Standard form: $p(y;\eta) = b(y) \exp(\eta^T T(y) - a(\eta))$
- The log-partition function $a(\eta)$ encodes the mean and variance

### 3. Generalized Linear Models (GLMs)
- **The recipe:** Choose distribution → response function falls out → model is determined
- **The elegant unification:** ALL GLMs share the same update rule form
- **Gaussian → Linear Regression** (identity response)
- **Bernoulli → Logistic Regression** (sigmoid response)
- **Poisson → Poisson Regression** (exponential response)
- **Multinomial → Softmax Regression** (softmax response)

### 4. Design Principle
When facing a new ML problem, ask:
1. What type of output am I predicting? (continuous, binary, counts, multi-class)
2. What distribution models that output?
3. Is that distribution in the exponential family?
4. If yes → GLM framework gives you the model automatically!